In [5]:
# 1. Montar Drive (seguro e idempotente)
from google.colab import drive
import os, subprocess

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

In [6]:
# 2. Definir caminhos
ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")

BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
RAW_PATH = os.path.join(BASE_PATH, "data_raw")
os.makedirs(RAW_PATH, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH :", RAW_PATH)

BASE_PATH: /content/drive/MyDrive/TCC_USP
RAW_PATH : /content/drive/MyDrive/TCC_USP/data_raw


In [7]:
# 3. Buscar notícias (NewsAPI)
import requests, pandas as pd
from datetime import datetime, timedelta

API_KEY = "3d606b02f24447849f49b3e8c3628f46"
URL = "https://newsapi.org/v2/everything"

params = {
    "q": "Ibovespa OR Bovespa OR Petrobras OR Vale OR dólar OR economia",
    "language": "pt",
    "sortBy": "publishedAt",
    "pageSize": 100,
    "apiKey": API_KEY
}

resp = requests.get(URL, params=params)
data = resp.json()

noticias = []
for art in data.get("articles", []):
    noticias.append({
        "data": art["publishedAt"][:10],  # formato YYYY-MM-DD
        "fonte": art["source"]["name"],
        "titulo": art["title"],
        "texto": art["description"] or ""
    })

df_new = pd.DataFrame(noticias)
print("Novas notícias coletadas:", df_new.shape)
display(df_new.head())

# 4. Incrementar dataset
out_file = os.path.join(RAW_PATH, "noticias_real.csv")

if os.path.exists(out_file):
    df_old = pd.read_csv(out_file)
    print("Dataset anterior:", df_old.shape)
else:
    df_old = pd.DataFrame(columns=["data","fonte","titulo","texto"])
    print("Nenhum dataset anterior encontrado, iniciando do zero.")

# Concatenar e remover duplicatas
df_full = pd.concat([df_old, df_new], ignore_index=True)
df_full = df_full.drop_duplicates(subset=["data","titulo"]).reset_index(drop=True)

# Salvar atualizado
df_full.to_csv(out_file, index=False, encoding="utf-8-sig")

print(f"✅ Atualização concluída! Total de notícias agora: {df_full.shape[0]}")

Novas notícias coletadas: (83, 4)


,data,fonte,titulo,texto
0,2025-09-27,Purepeople.com.br,"Ângela da novela 'Quatro por Quatro', Tatyane ...",Lançamento de 'Dindo Léo Pra Dedéu' reuniu ain...
1,2025-09-27,Viomundo.com.br,"Liszt Vieira: Donald Trump, química e mudanças",Química é econômica e geopolítica\nO post Lisz...
2,2025-09-27,Observador.pt,"""Não devíamos admitir quem tem forte influênci...","Bruno Mascarenhas, candidato do Chega à Câmara..."
3,2025-09-27,Terra.com.br,"Como visitante, Sport nunca venceu o Fortaleza...","Neste sábado (27), no Castelão, na capital cea..."
4,2025-09-27,Globo,Haddad diz que o grande problema do crime orga...,"Como resposta a esse cenário, Haddad voltou a ..."


Dataset anterior: (41, 4)
✅ Atualização concluída! Total de notícias agora: 81
